# Intersection / Overlap classifier — FIXED

Notebook corregido para entrenar un **modelo independiente de victim–perpetrator overlap** (`INTERSECT`) evitando leakage:

- Se crea `INTERSECT = (VÍCTIMA > 0) & (PERPETRADOR > 0)`.
- Se hace **train/test split antes de escalar y antes de PCA**.
- `MinMaxScaler`, medias de centrado y `PCA` se ajustan **solo con train**.
- Test se transforma con el scaler/PCA ya ajustado en train.
- Se usa el mismo criterio PCA de los otros modelos: **componentes hasta preservar ≥95% de varianza**.
- Se guardan predicciones, métricas, binarios TP/FP/TN/FN, varianza PCA y loadings.

Ejecutar después de tener disponibles:

- `./data/lista_global_vars.csv`
- `./data/target_col.csv`


In [ ]:

import os
import json
from pathlib import Path
import itertools
import random

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

import joblib
import tensorflow as tf
from tensorflow.keras import layers, models, mixed_precision
from tensorflow.keras.callbacks import EarlyStopping, Callback

# ============================================================
# Configuración general
# ============================================================
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

OUTPUT_DIR = Path("./content/intersection_fixed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PCA_VARIANCE_THRESHOLD = 0.95
TEST_SIZE = 0.25
BATCH_SIZE = 128

# Si quieres una búsqueda rápida para probar, pon FAST_GRID=True.
# Para replicar una búsqueda más amplia, deja FAST_GRID=False.
FAST_GRID = False

print("TensorFlow:", tf.__version__)
print("GPU available:", bool(tf.config.list_physical_devices("GPU")))
print("Output dir:", OUTPUT_DIR)


# ============================================================
# Helpers
# ============================================================
def report_to_dataframe(y_true, y_pred):
    """classification_report robusto para guardar como CSV sin romper Colab."""
    report_dict = classification_report(
        y_true,
        y_pred,
        digits=3,
        output_dict=True,
        zero_division=0,
    )

    rows = []
    for label, metrics in report_dict.items():
        if isinstance(metrics, dict):
            rows.append({
                "label": label,
                "precision": metrics.get("precision"),
                "recall": metrics.get("recall"),
                "f1-score": metrics.get("f1-score"),
                "support": metrics.get("support"),
            })
        else:
            rows.append({
                "label": label,
                "precision": None,
                "recall": None,
                "f1-score": metrics,
                "support": report_dict["weighted avg"]["support"],
            })
    return pd.DataFrame(rows)


def binary_metrics_dataframe(y_true, y_pred, label_name="intersection"):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    ppv = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan
    f1 = 2 * ppv * sensitivity / (ppv + sensitivity) if (ppv + sensitivity) > 0 else np.nan
    acc = (tp + tn) / (tp + fp + tn + fn)
    bal_acc = (sensitivity + specificity) / 2

    return pd.DataFrame([{
        "outcome": label_name,
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "precision_ppv": ppv,
        "recall_sensitivity": sensitivity,
        "specificity": specificity,
        "npv": npv,
        "f1_positive": f1,
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn),
        "support": int(tp + fp + tn + fn),
    }])


def fit_train_only_pca(X_train_df, X_test_df, threshold=0.95, output_dir=OUTPUT_DIR):
    """
    Fit MinMaxScaler + PCA SOLO con train y transforma train/test.
    Replica la lógica de los notebooks originales: MinMaxScaler -> centrar con medias -> PCA.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    scaler = MinMaxScaler(feature_range=(0, 1))
    X_train_scaled = scaler.fit_transform(X_train_df.values)
    X_test_scaled = scaler.transform(X_test_df.values)

    means_array = X_train_scaled.mean(axis=0)
    means = pd.Series(means_array, index=X_train_df.columns, name="mean")

    X_train_centered = X_train_scaled - means_array
    X_test_centered = X_test_scaled - means_array

    pca = PCA(n_components=X_train_df.shape[1], random_state=SEED)
    X_train_pca_all = pca.fit_transform(X_train_centered)
    X_test_pca_all = pca.transform(X_test_centered)

    cols_all = [f"PC{i+1}" for i in range(X_train_df.shape[1])]
    var_ratio = pca.explained_variance_ratio_
    cum_var = var_ratio.cumsum()
    df_variance = pd.DataFrame({
        "PC": cols_all,
        "explained variance": var_ratio,
        "cumulative variance": cum_var,
    })

    n_components = int(np.searchsorted(cum_var, threshold) + 1)
    cols = cols_all[:n_components]

    X_train_pca = pd.DataFrame(X_train_pca_all[:, :n_components], index=X_train_df.index, columns=cols)
    X_test_pca = pd.DataFrame(X_test_pca_all[:, :n_components], index=X_test_df.index, columns=cols)

    # Loadings completos y reducidos
    loadings = pd.DataFrame(
        pca.components_,
        columns=X_train_df.columns,
        index=cols_all,
    )

    top_rows = []
    for pc in cols:
        s = loadings.loc[pc].sort_values(key=np.abs, ascending=False).head(10)
        row = {"PC": pc}
        for i, (var, val) in enumerate(s.items(), start=1):
            row[f"var_{i}"] = var
            row[f"loading_{i}"] = val
        top_rows.append(row)
    top_loadings = pd.DataFrame(top_rows)

    # Guardar artefactos
    joblib.dump(scaler, output_dir / "scaler_minmax_train_only.pkl")
    joblib.dump(pca, output_dir / "pca_train_only.pkl")
    means.to_csv(output_dir / "pca_train_means.csv", encoding="utf-8-sig")
    df_variance.to_csv(output_dir / "df_PCA_variance_intersection.csv", index=False, encoding="utf-8-sig")
    loadings.to_csv(output_dir / "pca_loadings_intersection.csv", encoding="utf-8-sig")
    top_loadings.to_csv(output_dir / "pca_top_loadings_intersection.csv", index=False, encoding="utf-8-sig")
    X_train_pca.to_csv(output_dir / "X_train_pca.csv", index=True, encoding="utf-8-sig")
    X_test_pca.to_csv(output_dir / "X_test_pca.csv", index=True, encoding="utf-8-sig")

    print(f"PCA threshold: {threshold}")
    print(f"PCA components retained: {n_components}")
    print(f"Cumulative variance retained: {cum_var[n_components-1]:.4f}")

    return X_train_pca, X_test_pca, df_variance, loadings, top_loadings, scaler, pca, means


class OverfitStopping(Callback):
    def __init__(self, threshold=0.15):
        super().__init__()
        self.threshold = threshold

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        tr = logs.get("recall")
        vr = logs.get("val_recall")
        if tr is not None and vr is not None and (tr - vr) > self.threshold:
            print(f"\nDetenido en epoch {epoch+1}: Δrecall > {self.threshold}")
            self.model.stop_training = True


def train_overlap_nn(X_train_df, X_test_df, y_train_s, y_test_s, output_dir=OUTPUT_DIR, batch_size=128, fast_grid=False):
    """Entrena red neuronal para INTERSECT sobre PCA ya ajustado solo con train."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Estrategia TPU/GPU/CPU
    try:
        resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
        tf.config.experimental_connect_to_cluster(resolver)
        tf.tpu.experimental.initialize_tpu_system(resolver)
        strategy = tf.distribute.TPUStrategy(resolver)
        print("👾 TPU inicializado:", resolver.master())
    except Exception:
        strategy = tf.distribute.get_strategy()
        print("⚠️ No se encontró TPU, usando", type(strategy).__name__)

    # En GPU/CPU normal puede ir mejor float32 que bfloat16. En TPU, bfloat16 suele ir bien.
    try:
        if "TPUStrategy" in type(strategy).__name__:
            mixed_precision.set_global_policy("mixed_bfloat16")
        else:
            mixed_precision.set_global_policy("float32")
        print("Política de precisión:", mixed_precision.global_policy())
    except Exception as e:
        print("No se pudo ajustar mixed precision:", e)

    X_train = X_train_df.values.astype("float32")
    X_test = X_test_df.values.astype("float32")
    y_train = y_train_s.astype(int).values
    y_test = y_test_s.astype(int).values

    unique_classes, counts = np.unique(y_train, return_counts=True)
    class_weight = {int(cls): 1.0 for cls in unique_classes}
    if set(unique_classes) == {0, 1}:
        n0 = counts[unique_classes == 0].sum()
        n1 = counts[unique_classes == 1].sum()
        class_weight = {0: 1.0, 1: float(n0 / n1)}
    print("Class weight:", class_weight)

    train_ds = (
        tf.data.Dataset.from_tensor_slices((X_train, y_train))
        .shuffle(10000, seed=SEED)
        .batch(batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )
    test_ds = (
        tf.data.Dataset.from_tensor_slices((X_test, y_test))
        .batch(batch_size)
        .prefetch(tf.data.AUTOTUNE)
    )

    if fast_grid:
        grid_params = {
            "u1": [64, 32], "a1": ["relu", "tanh"], "d1": [0.3, 0.2],
            "u2": [16, 8],  "a2": ["relu", "tanh"], "d2": [0.1, 0.05],
        }
    else:
        grid_params = {
            "u1": [64, 32, 16], "a1": ["relu", "linear", "tanh", "sigmoid"], "d1": [0.3, 0.2, 0.1],
            "u2": [16, 8, 4],   "a2": ["relu", "linear", "tanh", "sigmoid"], "d2": [0.1, 0.05, 0.0],
        }

    thresholds = [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]

    results = []
    best = None
    best_model = None
    best_probs = None
    best_preds = None

    with strategy.scope():
        for u1, a1, d1, u2, a2, d2 in itertools.product(
            grid_params["u1"], grid_params["a1"], grid_params["d1"],
            grid_params["u2"], grid_params["a2"], grid_params["d2"]
        ):
            tf.keras.backend.clear_session()
            model = models.Sequential([
                layers.Input(shape=(X_train.shape[1],)),
                layers.Dense(u1, activation=a1),
                layers.Dropout(d1),
                layers.Dense(u2, activation=a2),
                layers.Dropout(d2),
                layers.Dense(1, activation="sigmoid", dtype="float32"),
            ])
            model.compile(
                optimizer=tf.keras.optimizers.Adam(1e-3),
                loss="binary_crossentropy",
                metrics=[
                    tf.keras.metrics.Recall(name="recall"),
                    tf.keras.metrics.Precision(name="precision"),
                    tf.keras.metrics.BinaryAccuracy(name="accuracy"),
                ],
            )

            model.fit(
                train_ds,
                validation_data=test_ds,
                epochs=200,
                class_weight=class_weight,
                callbacks=[
                    OverfitStopping(0.15),
                    EarlyStopping(monitor="val_recall", mode="max", patience=8, restore_best_weights=True),
                ],
                verbose=0,
            )

            probs = model.predict(test_ds, verbose=0).flatten()

            for thr in thresholds:
                preds = (probs >= thr).astype(int)
                tn, fp, fn, tp = confusion_matrix(y_test, preds, labels=[0, 1]).ravel()
                sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
                specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
                precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
                npv = tn / (tn + fn) if (tn + fn) > 0 else 0.0
                f1 = f1_score(y_test, preds, zero_division=0)
                acc = accuracy_score(y_test, preds)
                bal_acc = balanced_accuracy_score(y_test, preds)

                row = {
                    "u1": u1, "a1": a1, "d1": d1,
                    "u2": u2, "a2": a2, "d2": d2,
                    "threshold": thr,
                    "accuracy": acc,
                    "balanced_accuracy": bal_acc,
                    "recall_sensitivity": sensitivity,
                    "specificity": specificity,
                    "precision_ppv": precision,
                    "npv": npv,
                    "f1_positive": f1,
                    "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
                }
                results.append(row)

                # Criterio de selección para screening:
                # 1) prioriza recall >= .80; 2) maximiza balanced accuracy; 3) desempata por NPV y F1.
                eligible = sensitivity >= 0.80
                score_tuple = (
                    1 if eligible else 0,
                    bal_acc,
                    npv,
                    f1,
                    sensitivity,
                    specificity,
                )
                if best is None or score_tuple > best["score_tuple"]:
                    best = dict(row)
                    best["score_tuple"] = score_tuple
                    best_model = model
                    best_probs = probs.copy()
                    best_preds = preds.copy()

            print("Evaluado:", {"u1": u1, "a1": a1, "d1": d1, "u2": u2, "a2": a2, "d2": d2})

    df_results = pd.DataFrame(results)
    df_results.to_csv(output_dir / "gridsearch_results_intersection.csv", index=False, encoding="utf-8-sig")

    # Guardar mejor modelo y mejor configuración
    if best_model is not None:
        best_model.save(output_dir / "best_model_intersection.keras")

    best_public = {k: v for k, v in best.items() if k != "score_tuple"}
    with open(output_dir / "best_config_intersection.json", "w", encoding="utf-8") as f:
        json.dump(best_public, f, indent=2, ensure_ascii=False)

    # Guardar splits y matrices
    splits = {
        "train": X_train_df.index.astype(int).tolist(),
        "test": X_test_df.index.astype(int).tolist(),
    }
    with open(output_dir / "splits_indices.json", "w", encoding="utf-8") as f:
        json.dump(splits, f, indent=2)

    X_train_df.to_csv(output_dir / "X_train.csv", index=True, encoding="utf-8-sig")
    X_test_df.to_csv(output_dir / "X_test.csv", index=True, encoding="utf-8-sig")
    y_train_s.to_csv(output_dir / "y_train.csv", index=True, encoding="utf-8-sig")
    y_test_s.to_csv(output_dir / "y_test.csv", index=True, encoding="utf-8-sig")

    # Predicciones finales con el mejor threshold
    df_pred = pd.DataFrame({
        "idx_original": X_test_df.index,
        "y_true_intersection": y_test,
        "y_pred_intersection": best_preds.astype(int),
        "y_prob_intersection": best_probs,
        "threshold": best_public["threshold"],
    }).sort_values("idx_original")
    df_pred.to_csv(output_dir / "predictions_with_probs.csv", index=False, encoding="utf-8-sig")

    df_report = report_to_dataframe(y_test, best_preds)
    df_report.to_csv(output_dir / "results_intersection.csv", index=False, encoding="utf-8-sig")

    df_binary = binary_metrics_dataframe(y_test, best_preds, label_name="intersection")
    df_binary.to_csv(output_dir / "binary_metrics_intersection.csv", index=False, encoding="utf-8-sig")

    print("\n=== BEST CONFIG — INTERSECTION ===")
    print(pd.Series(best_public).to_string())

    print("\n=== FINAL REPORT — INTERSECTION ===")
    print(df_report.to_string(index=False))

    print("\n=== BINARY METRICS — INTERSECTION ===")
    print(df_binary.to_string(index=False))

    print("\nSaved files in:", output_dir)
    return df_results, best_public, df_pred, df_report, df_binary


# ============================================================
# 1) Carga y preparación de datos igual que el notebook original
# ============================================================
feat_df = pd.read_csv("./data/lista_global_vars.csv")
target_df = pd.read_csv("./data/target_col.csv").fillna(0)

print("Dim características:", feat_df.shape)
print("Dim target:", target_df.shape)
print("Valores faltantes:", feat_df.isna().sum().sum() + target_df.isna().sum().sum())
print("Target columns:", target_df.columns.to_list())

# Join + eliminación de categorías muy desbalanceadas, igual que en los otros notebooks.
df_merged = feat_df.join(target_df, how="inner")
df_merged = (
    df_merged[~((df_merged["GENERO_BIN_2"] == 1) | (df_merged["ORIENTSEX.BN_3"] == 1))]
    .drop(columns=["GENERO_BIN_2", "ORIENTSEX.BN_3"])
    .reset_index(drop=True)
)

# Target específico de overlap/intersección.
df_merged["INTERSECT"] = ((df_merged["VÍCTIMA"] > 0) & (df_merged["PERPETRADOR"] > 0)).astype(int)

print("\n=== CHECK TARGET INTERSECT EN MUESTRA COMPLETA ===")
print(df_merged["INTERSECT"].value_counts().sort_index())
print(df_merged["INTERSECT"].value_counts(normalize=True).sort_index())
print("Total analytical sample after filtering:", len(df_merged))
print("Total INTERSECT:", int(df_merged["INTERSECT"].sum()))

# Eliminar columnas de outcome/derivadas para evitar leakage conceptual.
drop_target_cols = [
    "VÍCTIMA", "PERPETRADOR", "VICTIMA_PERPETRADOR",
    "POLIVICTIMIZACION", "POLIPERPETRACION",
    "SOLO.VICTIMA", "SOLO.PERPETRADOR", "NO.VICT_NO.PERP",
    "V.O", "P.SUM.TOTAL", "V.SUM.TOTAL",
]
df_intersection = df_merged.drop(columns=drop_target_cols)
X = df_intersection.drop(columns=["INTERSECT"]).copy()
y = df_intersection["INTERSECT"].astype(int).copy()

# Recodificaciones idénticas al notebook de intersection original.
pd.set_option("future.no_silent_downcasting", True)

X["PAÍS"] = X["PAÍS"].replace({1: True, 2: False})
X["ETNIA.BN"] = X["ETNIA.BN"].replace({0.0: False, 1.0: True})
X["FUGAS.BN"] = X["FUGAS.BN"].replace({0.0: False, 1.0: True})

# Alternativa usada en el notebook original: convertir dummy de género/orientación a booleanos sin limpiar.
X["GENERO.BN0"] = X["GENERO_BIN_0"].replace({0.0: False, 1.0: True})
X["ORIENTSEX.BN0"] = X["ORIENTSEX.BN_1"].replace({0.0: False, 1.0: True})
X["GENERO.BN1"] = X["GENERO_BIN_1"].replace({0.0: False, 1.0: True})
X["ORIENTSEX.BN1"] = X["ORIENTSEX.BN_2"].replace({0.0: False, 1.0: True})
X = X.drop(columns=["GENERO_BIN_0", "GENERO_BIN_1", "ORIENTSEX.BN_1", "ORIENTSEX.BN_2"])

# Convive con hermanos / cero progenitores, igual que original.
X = X.rename(columns={"CONVIVEN.5": "CONVIVEN_H"})
X["CONVIVEN_H"] = X["CONVIVEN_H"].replace({0.0: False, 1.0: True})

X = X.rename(columns={"CONVIVEN.6": "CONVIVEN_0"})
X["CONVIVEN_0"] = X["CONVIVEN_0"].replace({0.0: False, 1.0: True})

# Asegurar numérico para scaler/PCA.
X = X.apply(pd.to_numeric, errors="coerce")
if X.isna().sum().sum() > 0:
    print("WARNING: Hay NaN tras conversión numérica. Se imputan a 0 para mantener el pipeline original.")
    print(X.isna().sum()[X.isna().sum() > 0])
    X = X.fillna(0)

X.to_csv(OUTPUT_DIR / "df_intersection_feat.csv", index=True, encoding="utf-8-sig")
y.to_csv(OUTPUT_DIR / "df_intersection_target.csv", index=True, encoding="utf-8-sig")

print("\nX shape:", X.shape)
print("y shape:", y.shape)
print("Feature columns:", X.columns.tolist())


# ============================================================
# 2) Split ANTES de scaler/PCA para evitar leakage
# ============================================================
idx_train, idx_test = train_test_split(
    X.index,
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=y,
)

X_train_raw = X.loc[idx_train].copy()
X_test_raw = X.loc[idx_test].copy()
y_train = y.loc[idx_train].copy()
y_test = y.loc[idx_test].copy()

print("\n=== CHECK SPLIT ===")
print("Train size:", len(idx_train), "Test size:", len(idx_test))
print("y_train value_counts:")
print(y_train.value_counts().sort_index())
print("y_test value_counts:")
print(y_test.value_counts().sort_index())


# ============================================================
# 3) PCA train-only, mismo criterio 95% que victim/perp
# ============================================================
X_train_pca, X_test_pca, df_variance, loadings, top_loadings, scaler, pca, means = fit_train_only_pca(
    X_train_raw,
    X_test_raw,
    threshold=PCA_VARIANCE_THRESHOLD,
    output_dir=OUTPUT_DIR,
)

print("\n=== TOP LOADINGS RETAINED PCS ===")
display(top_loadings)


# ============================================================
# 4) Entrenar modelo independiente de overlap/intersection
# ============================================================
df_results, best_config, df_pred, df_report, df_binary = train_overlap_nn(
    X_train_pca,
    X_test_pca,
    y_train,
    y_test,
    output_dir=OUTPUT_DIR,
    batch_size=BATCH_SIZE,
    fast_grid=FAST_GRID,
)
